# Lab 01 · Reference solution

The polished final implementation of the lab's agent loop:
- Provider-agnostic chat client (OpenAI or Anthropic via `PROVIDER`).
- Two tools (`calculator`, `web_search`) with Pydantic schemas.
- Agent loop with `MAX_STEPS` cap, structured tool errors, and
  consecutive-duplicate-action detection.

> ⏱ Read time: ~5 min · Notebook ~15 cells, deliberately shorter than the lab.
> 📖 The lab notebook (`../lab.ipynb`) is where the *reasoning* lives —
> why each piece is shaped this way, what failure modes it catches.
> This reference is the assembly. Compare against your own implementation
> after finishing the lab; come back to the lab for the why.

## Setup

In [ ]:
import json
import os
import pathlib
from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel, Field

# Walk up to find .env at the repo root (one level above this solution dir).
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)

PROVIDER = "openai"   # or "anthropic"
MODEL = {
    "openai": "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}[PROVIDER]

print(f"Using {PROVIDER} / {MODEL}")


**Sample output:**

```
Using openai / gpt-4o-mini
```

## Provider-agnostic chat client

One shape regardless of provider: input is OpenAI-flavored messages,
output is an `AssistantMessage` with `content` and `tool_calls`. The
Anthropic branch translates the tool schema and result envelope.

In [ ]:
@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict


@dataclass
class AssistantMessage:
    content: str | None
    tool_calls: list[ToolCall] = field(default_factory=list)


def chat_with_tools(
    messages: list[dict],
    tools: list[dict] | None = None,
    tool_choice: str = "auto",
) -> AssistantMessage:
    """Send messages, optionally with tools; return structured assistant response."""
    if PROVIDER == "openai":
        from openai import OpenAI

        resp = OpenAI().chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice=tool_choice if tools else None,
            temperature=0,
        )
        msg = resp.choices[0].message
        return AssistantMessage(
            content=msg.content,
            tool_calls=[
                ToolCall(
                    id=tc.id,
                    name=tc.function.name,
                    arguments=json.loads(tc.function.arguments),
                )
                for tc in (msg.tool_calls or [])
            ],
        )

    elif PROVIDER == "anthropic":
        from anthropic import Anthropic

        client = Anthropic()
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        anth_tools = [
            {
                "name": t["function"]["name"],
                "description": t["function"]["description"],
                "input_schema": t["function"]["parameters"],
            }
            for t in (tools or [])
        ]
        resp = client.messages.create(
            model=MODEL,
            system=system,
            messages=non_system,
            tools=anth_tools or None,
            max_tokens=1024,
            temperature=0,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tcs = [
            ToolCall(id=b.id, name=b.name, arguments=dict(b.input))
            for b in resp.content
            if getattr(b, "type", None) == "tool_use"
        ]
        return AssistantMessage(content=text or None, tool_calls=tcs)

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


## Tools

Two tools with Pydantic argument schemas. The calculator whitelists
characters before `eval` — a teaching example, not a sandbox. The web
search is mocked so the lab runs without a real search API key.

In [ ]:
# ── Calculator ────────────────────────────────────────────────────────────

class CalculatorArgs(BaseModel):
    expression: str = Field(
        description=(
            "Python arithmetic expression. Only numbers, +, -, *, /, %, "
            "parentheses, decimal points. Example: '(234 + 891 + 1502) / 3'"
        )
    )


def calculator(args: CalculatorArgs) -> dict:
    """Evaluate a simple arithmetic expression."""
    allowed = set("0123456789+-*/().%  ")
    if not set(args.expression) <= allowed:
        return {
            "error": "disallowed_characters",
            "detail": f"Only arithmetic allowed; got {set(args.expression) - allowed}",
        }
    try:
        return {"result": eval(args.expression, {"__builtins__": {}}, {})}  # noqa: S307
    except Exception as e:
        return {"error": type(e).__name__, "detail": str(e)}


# ── Mocked web search ─────────────────────────────────────────────────────

class WebSearchArgs(BaseModel):
    query: str = Field(description="Web search query.")


def web_search(args: WebSearchArgs) -> dict:
    """Mocked search; canned results keep the lab key-free."""
    canned = {
        "population of canada": "Canada's population is approximately 41 million (2024 estimate).",
        "population of france": "France's population is approximately 68 million (2024 estimate).",
        "population of japan": "Japan's population is approximately 125 million (2024 estimate).",
    }
    q = args.query.lower()
    for key, value in canned.items():
        if key in q:
            return {"results": [value]}
    return {"results": [], "note": "no canned result for this query"}


# ── Registry ──────────────────────────────────────────────────────────────

TOOLS: dict[str, tuple[Callable, type[BaseModel]]] = {
    "calculator": (calculator, CalculatorArgs),
    "web_search": (web_search, WebSearchArgs),
}


def tool_schemas() -> list[dict]:
    """OpenAI-format tool schemas built from the registry."""
    return [
        {
            "type": "function",
            "function": {
                "name": name,
                "description": (fn.__doc__ or "").strip().split("\n")[0],
                "parameters": args_model.model_json_schema(),
            },
        }
        for name, (fn, args_model) in TOOLS.items()
    ]


## Tool dispatcher

Convert any exception into structured data; the model can react to it
on the next step. Per [`concepts/tools/tool-design.md`](../../../concepts/tools/tool-design.md#4-the-return-contract):
errors are data, not crashes.

In [ ]:
def execute_tool(call: ToolCall) -> dict:
    """Dispatch a tool call to its handler; return structured result/error."""
    if call.name not in TOOLS:
        return {
            "error": "unknown_tool",
            "tool": call.name,
            "available": list(TOOLS),
        }
    fn, args_model = TOOLS[call.name]
    try:
        return fn(args_model.model_validate(call.arguments))
    except Exception as e:
        return {"error": type(e).__name__, "detail": str(e)}


## The agent loop

Three termination conditions:

1. **Final answer** — assistant returns content with no tool calls.
2. **Step cap** — `MAX_STEPS` exhausted.
3. **Repeated-action guard** — same `(tool_name, args)` signature
   twice in a row, twice in a row → halt. This is the canonical infinite-loop
   defense from [`concepts/agents/agent-loop.md`](../../../concepts/agents/agent-loop.md).

In [ ]:
MAX_STEPS = 8

REACT_SYSTEM_PROMPT = (
    "You are a tool-using assistant. For every step, follow this pattern:\n"
    "1. Write a brief thought (one or two sentences) about what to do next.\n"
    "2. Call the most appropriate tool, OR give a final answer if you have enough information.\n"
    "Be concise in both thoughts and final answers."
)


def _signature(call: ToolCall) -> str:
    """Stable fingerprint for repeated-action detection."""
    return f"{call.name}({json.dumps(call.arguments, sort_keys=True)})"


def run_agent(
    user_question: str,
    system_prompt: str = REACT_SYSTEM_PROMPT,
    verbose: bool = True,
) -> str:
    """Run the agent loop. Returns the final text answer (or halt notice)."""
    state: list[dict] = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question},
    ]
    last_sig: str | None = None
    duplicate_count = 0

    for step in range(MAX_STEPS):
        if verbose:
            print(f"\n── Step {step + 1} ──")

        msg = chat_with_tools(state, tools=tool_schemas())

        # Append assistant turn (with tool_calls if any)
        state.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)},
                }
                for tc in msg.tool_calls
            ] if msg.tool_calls else None,
        })

        # Termination 1: final answer
        if not msg.tool_calls:
            if verbose:
                print(f"Final: {msg.content}")
            return msg.content or ""

        # Execute each tool call; track signatures for dedup
        for call in msg.tool_calls:
            sig = _signature(call)
            if sig == last_sig:
                duplicate_count += 1
                if duplicate_count >= 2:
                    return f"[Halted: agent repeated the same action: {sig}]"
            else:
                duplicate_count = 0
            last_sig = sig

            if verbose:
                print(f"  → {call.name}({call.arguments})")
            result = execute_tool(call)
            if verbose:
                print(f"  ← {result}")

            state.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })

    return "[Halted: step cap reached]"


## Demo

A two-tool composition: compute, then check.

In [ ]:
answer = run_agent(
    "What is 17% of the average of 234, 891, and 1502? "
    "And is the result greater than 100?",
)
print(f"\n=== Final ===\n{answer}")


**Sample output (will vary slightly across runs):**

```
── Step 1 ──
  → calculator({'expression': '(234 + 891 + 1502) / 3'})
  ← {'result': 875.6666666666666}

── Step 2 ──
  → calculator({'expression': '0.17 * 875.6666666666666'})
  ← {'result': 148.86333333333332}

── Step 3 ──
Final: 17% of the average of 234, 891, and 1502 is approximately 148.86, which is greater than 100.

=== Final ===
17% of the average of 234, 891, and 1502 is approximately 148.86, which is greater than 100.
```

## Production readiness — out of scope here

For a real deployment you'd add: timeout-per-tool, total-time and total-token
budgets, async tool execution for parallel calls, structured logging,
per-tool circuit breakers, and observability hooks. The agent loop itself
doesn't change — those wrap around it. See the framework comparisons in
[`concepts/agents/agents-vs-frameworks.md`](../../../concepts/agents/agents-vs-frameworks.md)
for when reaching for LangGraph (Lab 05) pays off.